# warp-pack GPU validation

Goal: compare CPU vs GPU outputs and validate constraints/overlaps.

Steps:
1. Setup CUDA + Rust
2. Get the repo
3. Build `warp-pack` with CUDA
4. Run CPU + GPU
5. Validate outputs (min distance + constraint violations)

In [1]:
!nvidia-smi

Thu Feb  5 18:34:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential pkg-config
!curl https://sh.rustup.rs -sSf | sh -s -- -y
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ.get("PATH", "")
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
!rustc -V
!cargo -V

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,677 kB]
Get:13 http://security.ubuntu.com/ubuntu ja

## Get the repo

In [ ]:
import os
from getpass import getpass

# 1. This creates a secure input box. Paste your token there.
# It will not be visible and won't be saved in the notebook file.
token = getpass('Paste your GitHub Token: ')
os.environ['GITHUB_TOKEN'] = token

# 2. Now run your clone script
%cd /content
!rm -rf warp-md

REPO_URL = 'https://github.com/Haayhur/warp-md.git'
repo_host = REPO_URL.replace('https://', '')

# Use the token we just captured securely
if os.environ.get('GITHUB_TOKEN'):
    print("Token detected. Cloning...")
    # Using --quiet to prevent printing the URL with token in case of errors
    !git clone --depth 1 --quiet https://{os.environ['GITHUB_TOKEN']}@{repo_host} warp-md
else:
    !git clone --depth 1 {REPO_URL} warp-md

%cd warp-md

In [ ]:
%cd /content/warp-md
!cargo build -p warp-pack --features cuda --release

/content/warp-md
   Compiling proc-macro2 v1.0.106
   Compiling autocfg v1.5.0
   Compiling unicode-ident v1.0.22
   Compiling quote v1.0.44
   Compiling num-traits v0.2.19
   Compiling syn v2.0.114
   Compiling cfg-if v1.0.4
   Compiling libc v0.2.180
   Compiling zerocopy v0.8.38
   Compiling serde_core v1.0.228
   Compiling paste v1.0.15
   Compiling bytemuck v1.24.0
   Compiling safe_arch v0.7.4
   Compiling getrandom v0.2.17
   Compiling matrixmultiply v0.3.10
   Compiling typenum v1.19.0
   Compiling serde v1.0.228
   Compiling serde_derive v1.0.228
   Compiling ppv-lite86 v0.2.21
   Compiling rand_core v0.6.4
   Compiling wide v0.7.33
   Compiling num-integer v0.1.46
   Compiling num-complex v0.4.6
   Compiling approx v0.5.1
   Compiling rawpointer v0.2.1
   Compiling utf8parse v0.2.2
   Compiling simba v0.8.1
   Compiling anstyle-parse v0.2.7
   Compiling num-rational v0.4.2
   Compiling rand_chacha v0.3.1
   Compiling nalgebra-macros v0.2.2
   Compiling colorchoice v1.0.4
   C

In [ ]:
import json

def write_config(path, count, box_size, seed, min_distance=2.0, short_dist=1.0, short_scale=3.0, relax_steps=2, relax_step=0.5, gencan_maxit=20):
    cfg = {
      "box": {"size": [box_size, box_size, box_size], "shape": "orthorhombic"},
      "seed": seed,
      "min_distance": min_distance,
      "use_gencan": True,
      "gencan_maxit": gencan_maxit,
      "nloop0": 1,
      "nloop": 2,
      "use_short_tol": True,
      "short_tol_dist": short_dist,
      "short_tol_scale": short_scale,
      "relax_steps": relax_steps,
      "relax_step": relax_step,
      "structures": [
        {
          "path": "python/warp_md/pack/data/tip3p.pdb",
          "count": count,
          "rotate": True,
          "constraints": [
            {"mode": "inside", "shape": "sphere", "center": [box_size/2, box_size/2, box_size/2], "radius": box_size*0.45}
          ]
        }
      ]
    }
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2)
    print("Wrote", path)

# Baseline
write_config("pack_validate.json", count=100, box_size=40.0, seed=0)

# Large accuracy stress case (adjust count/box if too slow)
write_config("pack_validate_large.json", count=2000, box_size=70.0, seed=1, gencan_maxit=30, relax_steps=4)


Wrote pack_validate.json
Wrote pack_validate_large.json


In [ ]:
import os, subprocess

def run_pack(cfg_path, prefix, use_gpu):
    env = os.environ.copy()
    if use_gpu:
        env["WARP_PACK_USE_GPU"] = "1"
    out_path = f"{prefix}.pdb"
    cmd = ["./target/release/warp-pack", "--config", cfg_path, "--output", out_path, "--format", "pdb"]
    print("Running:", " ".join(cmd), "GPU" if use_gpu else "CPU")
    subprocess.run(cmd, check=True, env=env)
    return out_path

# Baseline
cpu_pdb = run_pack("pack_validate.json", "cpu", use_gpu=False)
gpu_pdb = run_pack("pack_validate.json", "gpu", use_gpu=True)

# Large
cpu_large = run_pack("pack_validate_large.json", "cpu_large", use_gpu=False)
gpu_large = run_pack("pack_validate_large.json", "gpu_large", use_gpu=True)


Running: ./target/release/warp-pack --config pack_validate.json --output cpu.pdb --format pdb CPU
Running: ./target/release/warp-pack --config pack_validate.json --output gpu.pdb --format pdb GPU
Running: ./target/release/warp-pack --config pack_validate_large.json --output cpu_large.pdb --format pdb CPU
Running: ./target/release/warp-pack --config pack_validate_large.json --output gpu_large.pdb --format pdb GPU


In [ ]:
import numpy as np

def load_pdb_coords(path):
    coords = []
    with open(path, "r") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])
    return np.array(coords, dtype=np.float32)

def min_pair_distance(coords):
    n = coords.shape[0]
    if n < 2:
        return 0.0
    min_d2 = np.inf
    for i in range(n - 1):
        diff = coords[i+1:] - coords[i]
        d2 = np.sum(diff * diff, axis=1)
        if d2.size:
            min_d2 = min(min_d2, float(d2.min()))
    return float(np.sqrt(min_d2))

def max_sphere_violation(coords, center, radius):
    diff = coords - np.array(center, dtype=np.float32)
    dist = np.sqrt(np.sum(diff * diff, axis=1))
    v = dist - radius
    return float(np.maximum(v, 0.0).max())

def compare(cpu_path, gpu_path, center, radius):
    cpu = load_pdb_coords(cpu_path)
    gpu = load_pdb_coords(gpu_path)
    print("CPU atoms:", cpu.shape[0], "GPU atoms:", gpu.shape[0])
    min_dist_cpu = min_pair_distance(cpu)
    min_dist_gpu = min_pair_distance(gpu)
    max_v_cpu = max_sphere_violation(cpu, center, radius)
    max_v_gpu = max_sphere_violation(gpu, center, radius)
    print("min distance CPU:", min_dist_cpu)
    print("min distance GPU:", min_dist_gpu)
    print("max sphere violation CPU:", max_v_cpu)
    print("max sphere violation GPU:", max_v_gpu)
    if cpu.shape == gpu.shape:
        diff = np.abs(cpu - gpu)
        print("coord max abs diff:", float(diff.max()))
        print("coord mean abs diff:", float(diff.mean()))

print("Baseline")
compare("cpu.pdb", "gpu.pdb", [20.0, 20.0, 20.0], 18.0)

print("Large")
# box_size=70.0 => center [35,35,35], radius 31.5
compare("cpu_large.pdb", "gpu_large.pdb", [35.0, 35.0, 35.0], 31.5)


Baseline
CPU atoms: 300 GPU atoms: 300
min distance CPU: 0.9559489455696075
min distance GPU: 0.9559489455696075
max sphere violation CPU: 0.0
max sphere violation GPU: 0.0
coord max abs diff: 0.0
coord mean abs diff: 0.0
Large
CPU atoms: 6000 GPU atoms: 6000
min distance CPU: 0.9557069607821964
min distance GPU: 0.9557069607821964
max sphere violation CPU: 0.0
max sphere violation GPU: 0.0
coord max abs diff: 0.0
coord mean abs diff: 0.0


In [ ]:
##Baseline
CPU atoms: 300 GPU atoms: 300
min distance CPU: 0.9559489455696075
min distance GPU: 0.9559489455696075
max sphere violation CPU: 0.0
max sphere violation GPU: 0.0
coord max abs diff: 0.0
coord mean abs diff: 0.0
Large
CPU atoms: 6000 GPU atoms: 6000
min distance CPU: 0.9557069607821964
min distance GPU: 0.9557069607821964
max sphere violation CPU: 0.0
max sphere violation GPU: 0.0
coord max abs diff: 0.0
coord mean abs diff: 0.0